# Session 02: Tokenization and Embeddings

**Student implementer and maintainer:** YQ-HA. This notebook records code typed,
executed, and adapted while following the instructor during class, together with
the student's post-course completions and maintenance.

**Course:** Tyler Shoemaker, [Introduction to Interpretability for Language Models](https://digitaltheorylab.github.io/2026_interpretability-reader/).
Reference implementation: [02_inputs.md](https://github.com/digitaltheorylab/2026_interpretability-reader/blob/main/chapters/02_inputs.md).

**Repair assistance:** OpenAI Codex assisted with locating and explaining errors,
restoring omitted course code, and checking coverage and runnability. `Repair:`
comments identify corrected mistakes. Original classroom notes, including `##`
comments, are retained.

Run from the first cell with the kernel in `notebooks/`. New results go to
`results/session-02/`. This repaired version completed its Colab execution on October 26, 2026. See `results/` for the saved run; session 08 analyzed a truncated reasoning trace.


In [ ]:
from pathlib import Path
import json
import platform
import sys
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "session-02"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FACET_HEIGHT = 4
FACET_ASPECT = 1.2
def save_figure(name):
    plt.gcf().savefig(RESULTS_DIR / name, bbox_inches="tight", dpi=150)
    plt.show()
    plt.close("all")


In [ ]:

from heapq import nlargest

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModelForCausalLM, AutoTokenizer



In [ ]:
checkpoint = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint, low_cpu_mem_usage=True)
model.eval()



In [ ]:
# Repair: Move the embedding tensor to CPU before NumPy conversion.
embedding_layer = model.get_input_embeddings()
emb = embedding_layer.weight.detach().cpu().numpy().astype(np.float64)
print("Embedding matrix shape:", emb.shape)   # (50257, 768)


SQUARE_FIGSIZE = 6

In [ ]:
# 
encoded = tokenizer.encode("Hello")
print("Token ID:", encoded)          # [15496]
print("Decoded:", tokenizer.decode(encoded))

# 
encoded = tokenizer.encode("Hello world!")
print("Token IDs:", encoded)         # [15496, 995, 0]
print("First token:", tokenizer.decode(encoded[0]))
print("Last token:", tokenizer.decode(encoded[-1]))

# 
encoded = tokenizer.encode("programmatic")
print("Subword tokens:", encoded)    # [15385, 1070] -> "program" + "matic"

# 
text = "large language models use subword tokenization"
encoded = tokenizer.encode(text)
print("Number of tokens:", len(encoded))   # 8

# 
for number in encoded:
    decoded = tokenizer.decode(number)
    print(f"{number:>6} --> {repr(decoded)}")

In [ ]:
texts = ["Token", "Token", "token"]
encoded_batch = tokenizer(texts)["input_ids"]
print(list(zip(texts, encoded_batch)))
assert encoded_batch[0] == encoded_batch[1]
assert encoded_batch[0] != encoded_batch[2]


In [ ]:
assert emb.shape[0] == model.config.vocab_size

vector = emb[0]
print("Vector length:", len(vector))   # 768

encoded = tokenizer.encode("Hello world!")
for number in encoded:
    vector = emb[number]
    print(f"Token ID {number} -> vector length {len(vector)}")

emb_seq = emb[encoded]
print("Sequence embeddings shape:", emb_seq.shape)   # (3, 768)

In [ ]:
A = emb[0]
B = emb[1]

magnitude = np.linalg.norm(A)
print("Magnitude of A:", round(magnitude, 2))

direction = A / magnitude
print("Direction vector length:", len(direction))
print("Magnitude of direction:", round(np.linalg.norm(direction), 2))   # 1.0

In [ ]:
# 
dot = np.dot(A, B)   
print("Dot product:", round(dot, 2))

# 
cos_sim = dot / (np.linalg.norm(A) * np.linalg.norm(B))
print("Cosine similarity:", round(cos_sim, 2))

# 
sim_matrix = cosine_similarity([A, B])   # 2x2 
print("Similarity matrix:\n", np.round(sim_matrix, 3))

In [ ]:
# 
query_enc = tokenizer.encode(" language")
query_vec = emb[query_enc]   # 

#
scores = cosine_similarity(query_vec, emb)[0]   # 

# 
top_k = 25
scored = [(scores[i], i) for i in range(len(scores))]
top = nlargest(top_k, scored)

print("Top 25 tokens most similar to 'language':")
for score, idx in top:
    token = tokenizer.decode(idx)
    print(f"{repr(token):<15} {score:.3f}")

In [ ]:
words = " bank money loan river stream water bark tree dog cat"
encoded = tokenizer.encode(words)
batch_emb = emb[encoded]   # (10, 768)

# 
sim = cosine_similarity(batch_emb)   # (10, 10)

# 
labels = [tokenizer.decode(tok) for tok in encoded]

# 
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(sim, annot=True, fmt=".2f", cmap="RdBu_r",
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("Cosine Similarity among Selected Words")
save_figure("s02-original-008.png")

In [ ]:
# Repair: Seed the PCA projection for repeatable classroom plots.
# 
pca = PCA(n_components=2, random_state=5167)
emb_2d = pca.fit_transform(batch_emb)

# 
fig, ax = plt.subplots(figsize=(SQUARE_FIGSIZE, SQUARE_FIGSIZE))
sns.scatterplot(x=emb_2d[:, 0], y=emb_2d[:, 1], ax=ax)
for i, label in enumerate(labels):
    ax.text(emb_2d[i, 0], emb_2d[i, 1], label, fontsize=9)
ax.set_title("PCA Projection of Word Embeddings")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
save_figure("s02-original-009.png")

In [ ]:
observed = {name: module.__version__ for name, module in sys.modules.items()
            if name in {"numpy", "pandas", "torch", "transformers", "sklearn", "scipy", "matplotlib", "seaborn"}}
environment = {"python": platform.python_version(), "imported_packages": observed}
(RESULTS_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print("Completed notebook; results:", RESULTS_DIR)
